In [1]:
# TODAY WE FINALLY CLOSE MONTH 2: THE RECALL TEST + MONTH 2 REVIEW

"""
This closes week 8, and closes month 2 entirely. I now have four memory tiers wired into one graph (Day 24's entity_graph): a 10-message window,
a compresses Redis summary, durable MongoDB profile facts, and mergeable MongoDB entities. Today doesn't add a fifth tier. It does something more important: 
it proves the first four actually work together, with a scripted multi-session conversation designed so that facts planted early can only be correctly recalled later if the right tier fires 
at the right time. Then it closes out Month 2 with architecture documentation and a system-wide integration test spanning everything built since day 7.

Deep Coding: The recall Test Harness
- Scripted Fact Placement: A conversation script where specific facts are deliberately planted to land in specific tiers - one fact stated in turn 1 (will bw pushed into the summary by turn 7's overflow),
one stated as an explicit standing preference (should become a profile fact), one tied to a named recurring topic (should become a tracked entity), and one stated in the last few turns (should still be in the raw window).

- Recall Queries with No Restated Context: A second batch of questions, asked later, the deliberately omit any restatement of the planted facts- if the system answers them correctly, it can only be because the right memory
tier supplied the missing context.

- An LLM Recall Judge: Since "did the answer correctly use memory" isn't checkable with a keyword match (Day 13's blunt approach), build a dedicated judge prompt that compares an answer against an explicit expect_recall string and returns PASS/FAIL with a reason
same pattern as day 11's reviewer, applied to memory instead of retrieval faithfulness.

- Failure Triage Guide: When a recall query fails, the fix depends entirely on which tier was supposed to supply the answer. Build a smalldiagnostic function that inspects Redis and MongoDB directly for a given session_id/user_id to show you exactly what each tier actually held
at the time of failure - not what it should have held.

"""

'\nThis closes week 8, and closes month 2 entirely. I now have four memory tiers wired into one graph (Day 24\'s entity_graph): a 10-message window,\na compresses Redis summary, durable MongoDB profile facts, and mergeable MongoDB entities. Today doesn\'t add a fifth tier. It does something more important: \nit proves the first four actually work together, with a scripted multi-session conversation designed so that facts planted early can only be correctly recalled later if the right tier fires \nat the right time. Then it closes out Month 2 with architecture documentation and a system-wide integration test spanning everything built since day 7.\n\nDeep Coding: The recall Test Harness\n- Scripted Fact Placement: A conversation script where specific facts are deliberately planted to land in specific tiers - one fact stated in turn 1 (will bw pushed into the summary by turn 7\'s overflow),\none stated as an explicit standing preference (should become a profile fact), one tied to a named 

In [2]:

# Importing necessary libraries


import os
import re
import json
from groq import Groq
from dataclasses import dataclass, field, asdict
from datetime import datetime
from typing import Optional, TypedDict, Literal
from langgraph.graph import StateGraph, END
from dotenv import load_dotenv, find_dotenv
from dotenv import load_dotenv, find_dotenv
from llama_index.core import Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.mongodb import MongoDBAtlasVectorSearch
from llama_index.storage.docstore.mongodb import MongoDocumentStore
from pymongo import MongoClient, AsyncMongoClient
from llama_index.core import VectorStoreIndex, StorageContext
from langchain_community.tools.tavily_search import TavilySearchResults

Settings.embed_model = HuggingFaceEmbedding(
    model_name = 'BAAI/bge-m3'
)


load_dotenv(find_dotenv())

client = Groq()
webSearch = TavilySearchResults(max_results = 3)


# LOADING EXISTING VECTOR STORE

# first connecting to existing vector store
# connecting to existing vectorstore
# Re-establishing the connection to my mongoDB atlas vector store to retrieve historical data

# mongoDB atlas connection
mongoClient = MongoClient(os.getenv('MONGO_URI'))
asyncMongoClient = AsyncMongoClient(os.getenv('MONGO_URI'))


# reconnecting persistent docstore
docstore = MongoDocumentStore.from_uri(
    uri = os.getenv('MONGO_URI'),
    db_name = 'month1_database',
    namespace = 'month_1_collection'
)

# 1. Connect to your MongoDB
vectorStore = MongoDBAtlasVectorSearch(
    mongodb_client=mongoClient,
    async_mongodb_client= asyncMongoClient, 
    db_name= 'month1_database',
    collection_name= 'month_1_rag_collection_v3',
    vector_index_name= 'final_index_v3',
    embedding_key= 'embedding'
)

storageContext = StorageContext.from_defaults(
    vector_store= vectorStore,
    docstore= docstore
)

index = VectorStoreIndex.from_vector_store(
    vector_store= vectorStore,
    storage_context = storageContext
)

# creating the primary retrieval tool
'''apple_10k_expert = QueryEngineTool(
    query_engine= index.as_query_engine(similarity_top_k = 15),
    metadata= ToolMetadata(
        name = 'apple_10k_expert',
        description= "Search Through Apple's 10-K filings for historical financial data, revenue figures, and risk factors."
    )

)'''

print('🤖🛩️ Vector Store connection established ⚡')


# TOKEN USER TRACKER
@dataclass
class TokenUsage:
    prompt_tokens: int = 0
    completion_tokens: int = 0
    total_calls: int = 0

    def add(self, usage):
        self.prompt_tokens += usage.prompt_tokens
        self.completion_tokens += usage.completion_tokens
        self.total_calls += 1
    
    def cost_estimate(
            self,
            input_price_per_1m: float = 2.50,
            output_price_per_1m: float = 10.00
    ) -> float:
        input_cost = (self.prompt_tokens /1000000) * input_price_per_1m
        output_cost = (self.completion_tokens /1000000) * output_price_per_1m
        return round(input_cost + output_cost, 6)
    
    def report(self):
        print(f"\n📣 Token usage report")
        print(f"LLM calls : {self.total_calls}")
        print(f"Prompt tokens: {self.prompt_tokens}")
        print(f"Completion tokens: {self.completion_tokens}")
        print(f"Total tokens: {self.prompt_tokens + self.completion_tokens:,}")
        print(f"Est. cost (GPT-40 pricing): ${self.cost_estimate()}")


# global tracher that is set before each run
usage_tracker = TokenUsage()


# TOKEN aware LLM Caller

def tracked_llm_call(messages: list, system: str = "") -> str:
    """
    Wraps every Groq call so token usage is always captured.
    Drop-in replacement for direct client.chat.completions.create calls.
    
    """

    full_messages = []
    if system:
        full_messages.append({"role": "system", "content": system})
    full_messages.extend(messages)

    response = client.chat.completions.create(
        model = "openai/gpt-oss-120b",
        messages= full_messages,
        temperature= 0,
    )

    usage_tracker.add(response.usage)
    return response.choices[0].message.content

# RETRIEVAL WITH CONFIDENCE SCORING
RELEVANCE_THRESHOLD = 0.5

def retrieve_with_confidence(query: str) -> tuple[list, float]:
    """Returns retrieved docs and the top chunk's confidence score.
    Uses cosine similarity to score from your vectore store.
    """

    # creating a native llamaindex retriever from my initialized index
    retriever = index.as_retriever(similarity_top_k = 4)
    results = retriever.retrieve(query)

    # reults is a list of (document, score) tuples
    # lower score = more similar in FAISS  (L2 distance); invert if needed
    # For Cosine similarity stores, higher = better

    if not results:
        return [], 0.0
    
    
    top_score = float(results[0].score) if results[0].score is not None else 0.0

    print(f"📣 Top retrieval score : {top_score:.3f} (threshold: {RELEVANCE_THRESHOLD})")
    return results, top_score

def format_chunks(docs: list) -> str:
    return "\n\n---\n\n".join([doc.node.get_content() for doc in docs])
        


# CRAG -> Retrieval quality gate

def corrective_retrieve(query: str) -> tuple[str, str]:
    """
    Returns (context_text, source) where source is 'local' or 'web'.
    Applies CRAG logic: low confidence -> discard local, use web fallback.
    """

    docs, top_score = retrieve_with_confidence(query)

    if top_score < RELEVANCE_THRESHOLD or not docs:
        print("🤥 CRAG: Low retrieval confidence - falling back to web search")
        webResults = webSearch.invoke(query)
        context = "\n\n".join([r["content"] for r in webResults])
        return context, "web"
    
    else:
        print("👍 CRAG: Retrieval confidence acceptable - using local docs")
        return format_chunks(docs), "local"
    

GENERATOR_SYSTEM_PROMPT = """You are a precise financial analyst assistant.
Answer the user's question using ONLY the context provided.
If the context does not contain eough information to answer, say exactly:
'I cannot find sufficient information in the provided context.' 
Be specific - include numbners, percentages, and fiscal year references where available.

"""

REVIEWER_SYSTEM_PROMPT = """You are a strict factual reviewer for a financial RAG system.
You will receive a question, the source context, and a generated answer.

Your job is to check:
1. Does the answer contain any claims NOT supported by the context? (hallucination)
2. Does the answer actually address the question asked?
3. Are numbers, percentages, and figures accurate relative to the context?

Respond in EXACTLY this format:
Verdict: <PASS or FAIL>
Reason: <one sentence explaining your verdict>

PASS meaans the answer is faithful to the context and addresses the question,
FAIL means the answer contains unsupported claims, wrong figuress, or avoids the question.


"""

ROUTER_SYSTEM_PROMPT = """You are a query complexity classifier for a financial RAG system.

Classify the user's question as one of:
- SIMPLE: a single factual lookup requiring one retrieval and one answer
(e.g. "What was Apple's net income in FY2024?")
- COMPLEX: requires multiple steps, comparisons, calculations, or chaining
(e.g. "Compare iPhone revenue across FY2023 and FY2024 and calculate the growth rate")
- UNKNOWN: cannot be answered from a financial document at all 
(e.g. "What is the weather in Cupertino today?")

Respond in EXACTLY this format:
Classification: <SIMPLE, COMPLEX, or UNKNOWN>
Reason: <one sentence>
"""

def generate_answer(question: str, context: str, critique: str = "") -> str:
    critiqueBlock = ""
    if critique:
        critiqueBlock = f"\n\nPrevious answer was rejected for this reason: {critique}\nPlease rewrite addressing this critique."

    response = client.chat.completions.create(
        model = "openai/gpt-oss-120b",
        messages = [
            {"role": "system", "content": GENERATOR_SYSTEM_PROMPT},
            {"role": "user", "content": (
                f"Context:\n{context}\n\n"
                f"Question: {question}"
                f"{critiqueBlock}"
            )}
        ],

        temperature= 0,
    )

    return response.choices[0].message.content


def review_answer(question: str, context: str, answer: str) -> tuple[str, str]:
    """Returns (verdict, reason) where verdict is a PASS or FAIL."""
    response = client.chat.completions.create(
        model = 'openai/gpt-oss-120b',
        messages = [
            {"role": "system", "content": REVIEWER_SYSTEM_PROMPT},
            {"role": "user", "content":(
                f"Question: {question}\n\n"
                f"Source Context:\n{context}\n\n"
                f"Generated Answer:\n{answer}"

            )}
        ],
        temperature= 0
    )

    raw = response.choices[0].message.content
    verdict_match = re.search(r"Verdict:\s*(PASS|FAIL)", raw)
    reason_match =re.search(r"Reason:\s*(.+)", raw)

    verdict = verdict_match.group(1) if verdict_match else "FAIL"
    reason = reason_match.group(1).strip() if reason_match else raw.strip()

    print(f"😮‍💨 Reviewer verdict: {verdict} - {reason}")
    return verdict, reason

# NAIVE RAG PATH 

NAIVE_RAG_SYSTEM_PROMPT = """You are a precise financial analyst assistant.
Answer the question uning ONLY THE context provided.
Be specific - include exact figures, percentages, and fiscal year references.
If the context does not contain the answer, say so directly.

"""

def run_naive_rag(question: str) -> dict:
    print("\n⚡ Path: NAIVE RAG")
    started_at = datetime.now().isoformat()

    # single retrieval
    docs, score = retrieve_with_confidence(question)
    context = "\n\n --- \n\n".join([doc.node.get_content() for doc in docs])

    # Single generation - no reviewer, no rewrite
    answer = tracked_llm_call(
        messages=[{
            "role": "user",
            "content": f"Context:\n{context}\n\nQuestion: {question}"
        }],
        system = NAIVE_RAG_SYSTEM_PROMPT
    )

    print(f"Answer: {answer}")
    return {
        "path": "naive_rag",
        "question": question,
        "answer": answer,
        "retrieval_score": score,
        "started_at": started_at,
        "finished_at": datetime.now().isoformat()
    }


# THE FULL SELF CORRECTING CRAG PIPELINE

def run_crag_pipeline(question: str, max_rewrites: int = 2) ->dict:
    print(f"\n{'='*60}")
    print(f"Question: {question}")
    print(f"\n{'='*60}")

    startedAt = datetime.now().isoformat()

    # STEP 1: CRAG retrieval with confidence gate
    context, source = corrective_retrieve(question)

    # STEP 2: Generate Initial Answer
    print("\n 🦾 Generating initial answer...")
    answer = generate_answer(question, context= context)
    print(f"Answer: {answer}\n")

    # SECONDARY GATE to catch false-positive vector scores
    if answer and "I cannot find sufficient information" in answer and source == "local":
        print("🤥 CRAG: Local docs failed to answer despite high vector score. Forcing web fallback...")
        webResults = webSearch.invoke(question)
        context = "\n\n".join([r["content"] for r in webResults])
        source = "web"

        print("👍Generating answer from web context...")
        answer = generate_answer(question, context= context)
        print(f"Web Fallback Answer: {answer}\n")

    # STEP 3: Reviewer Loop
    attempts = 0
    verdict = 'FAIL'
    critique = ""
    history = []

    while verdict == 'FAIL' and attempts < max_rewrites:
        verdict, critique = review_answer(question= question, context= context, answer= answer)
        history.append({
            "attempt": attempts +1,
            "answer": answer,
            "verdict": verdict,
            "critique": critique
        })

        if verdict == 'FAIL':
            attempts += 1
            if attempts < max_rewrites:
                print(f"\n🔁 Rewriting (attempt {attempts})...")
                answer = generate_answer(question, context, critique)
                print(f"Rewritten Answer: {answer}\n")
            else:
                print("🤥 Max rewrites reached - returning best attempt with warning")

    # final evrdict check if we exited the loop with PASS 
    if verdict != 'FAIL':
        verdict, critique = review_answer(question, context, answer)
        history.append({
            "attempt": attempts +1,
            "answer": answer,
            "verdict": verdict,
            "critique": critique
        })

    result = {
        "question": question,
        "retrieval_source": source,
        "final_answer": answer,
        "fianl_verdict": verdict,
        "rewrite_attempts": attempts,
        "review_history": history,
        "started_at":startedAt,
        "finished_at": datetime.now().isoformat()
    }


    print(f"\n{'='*60}")
    print(f"👍 Final Answer ({verdict} after {attempts} rewrite(s)):")
    print(answer)
    print(f"{'='*60}\n")

    filename = f"traces/day11_crag_{datetime.now().strftime('%H%M%S')}.json"
    with open(filename, 'w') as f:
        json.dump(result, f, indent= 2)
    print(f"📀 Saved to {filename}")

    return result


def classify_query(question: str) -> tuple[str, str]:
    raw = tracked_llm_call(
        messages = [{"role": "user", "content": f"Question: {question}"}],
        system= ROUTER_SYSTEM_PROMPT
    )
    classification_match = re.search(r"Classification:\s*(SIMPLE|COMPLEX|UNKNOWN)", raw)
    reason_match = re.search(r"Reason:\s*(.+)", raw)

    classification = classification_match.group(1) if classification_match else "COMPLEX"
    reason = reason_match.group(1).strip() if reason_match else "Could not Parse reason."

    print(f"🦾 Router: {classification} - {reason}")
    return classification, reason

# MULTI-PATH ROUTER

def run_router(question: str) -> dict:
    global usage_tracker
    usage_tracker = TokenUsage() # reset for each question

    print(f"\n{'='*60}")
    print(f"Question: {question}")
    print(f"{'='*60}")

    classification, reason = classify_query(question)

    if classification== 'SIMPLE': 
        result = run_naive_rag(question= question)

    elif classification == "COMPLEX":
        print("\n🤖 Path: AGENTIC RAG (CRAG + Reviewer)")
        result = run_crag_pipeline(question= question)
        result['path'] = "agentic_rag"
    
    else: # UNKNOWN
        print("\n🤥 Path: UNKNOWN - cannot answer from financial documents")
        result = {
            "path": "unknown",
            "question": question,
            "answer": "This question cannot be answered using the Apple 10-K document.",
            "started_at": datetime.now().isoformat(),
            "finished_at": datetime.now().isoformat()
        }
    
    usage_tracker.report()

    result["Classification"] = classification
    result["classification_reason"] = reason
    result["token_usage"] = {
        "prompt_tokens": usage_tracker.prompt_tokens,
        "completion_tokens": usage_tracker.completion_tokens,
        "total_calls": usage_tracker.total_calls,
        "estimated_cost_usd": usage_tracker.cost_estimate()
    }

    filename = f"traces/day12_{classification.lower()}_{datetime.now().strftime('%H%M%S')}.json"
    with open(filename, "w") as f:
        json.dump(result, f, indent = 2)
    print(f"\n📀 Saved to {filename}")

    return result


# Redis must be running locally before this cell.
# Option A (native): install Redis and run `redis-server` in a terminal
# Option B (Docker): docker run -d --name redis-memory -p 6379:6379 redis:latest

import redis
#import json

REDIS_HOST = "localhost"
REDIS_PORT = 6379
WINDOW_SIZE = 10  # keeps the last 10 messages = 5 user/assistant exchanges

redis_client = redis.Redis(
    host=REDIS_HOST, port=REDIS_PORT, db=0, decode_responses=True
)

def check_redis_available() -> bool:
    try:
        redis_client.ping()
        return True
    except redis.exceptions.ConnectionError:
        print(f"❌ Redis not reachable at {REDIS_HOST}:{REDIS_PORT}")
        print(f"   Start it with 'redis-server' or via Docker before continuing.")
        return False

print("✅ Redis client configured" if check_redis_available() else "⚠️  Redis unavailable — memory will be skipped")


# MONGODB connection for long-term profiles

from pymongo import MongoClient
from datetime import datetime

mongo_client = MongoClient(os.getenv("MONGO_URI"))
profile_db = mongo_client["user_memory"]
profiles_collection = profile_db["user_profiles"]

# Index for fast lookup by user_id
profiles_collection.create_index("user_id")

print("✅ MongoDB long-term profile store connected")


c:\Users\rodne\miniconda3\envs\rag-Ai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\rodne\AppData\Local\Temp\ipykernel_24020\474458759.py:30: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  webSearch = TavilySearchResults(max_results = 3)


🤖🛩️ Vector Store connection established ⚡
✅ Redis client configured
✅ MongoDB long-term profile store connected


In [3]:
class RAGState(TypedDict):
    question: str
    session_id: str
    user_id: str

    conversation_history: list[dict]
    conversation_summary: Optional[str]
    user_profile_facts: list[dict]
    tracked_entities: list[dict]           # NEW — structured analysis threads

    classification: Optional[str]
    classification_reason: Optional[str]
    context: Optional[str]
    retrieval_source: Optional[str]
    retrieval_score: Optional[float]
    answer: Optional[str]
    reviewer_verdict: Optional[str]
    reviewer_reason: Optional[str]
    rewrite_count: int
    max_rewrites: int
    sufficiency_verdict: Optional[str]
    sufficiency_reason: Optional[str]
    retrieval_attempts: int
    max_retrieval_attempts: int
    reformulated_query: Optional[str]
    api_error: bool
    api_error_message: Optional[str]
    api_error_type: Optional[str]
    using_fallback_llm: bool
    error_log: list[dict]
    newly_extracted_fact: Optional[str]
    newly_extracted_entity: Optional[dict]  # NEW
    warning: Optional[str]
    finished_at: Optional[str]

In [4]:
# NODES AND SUPPORTING FUNCTIONS


#  NODES

# -----Node 1: Classify -------
def classify_node(state: RAGState) -> dict:
    print(f"\n[NODE: classify] Question: {state['question'][:60]}...")
    raw = tracked_llm_call(
        messages=[{"role": "user", "content": f"Question: {state['question']}"}],
        system= ROUTER_SYSTEM_PROMPT
    )

    classification_match = re.search(r"Classification:\s*(SIMPLE|COMPLEX|UNKNOWN)", raw)
    reason_match = re.search(r"Reason:\s*(.+)", raw)

    classification = classification_match.group(1) if classification_match else "COMPLEX"
    reason = reason_match.group(1).strip() if reason_match else "Could not parse."

    print(f" -> Classification: {classification} - {reason}")
    return {"classification": classification, "classification_reason": reason}

# ------ Node 2: Retrieve ------ # LEAB=VING THIS ONE OUT AS WE ARE GOING TO WRITE A MODIFIED VERSION OF IT IN THE NEXT CELL


# ---Node 3: Generate ---- REPLACED TODAY


# -----Node 4: Review ----
def review_node(state: RAGState) -> dict:
    print(f"\n[NODE: review]")

    if state["classification"] == "UNKNOWN":
        print(" -> UNKNOWN path - skipping review")
        return {"reviewer_verdict": "PASS", "reviewer_reason": "Abstention accepted."}
    
    raw = tracked_llm_call(
        messages= [{
            "role": "user",
            "content": (
                f"Question: {state['question']}\n\n"
                f"Source Context:\n{state['context']}\n\n"
                f"Generated Answer:\n{state['answer']}"
            )
        }],
        system = REVIEWER_SYSTEM_PROMPT
    )

    verdict_match =re.search(r"Verdict:\s*(PASS|FAIL)", raw)
    reason_match = re.search(r"Reason:\s*(.+)", raw)

    verdict = verdict_match.group(1) if verdict_match else "FAIL"
    reason = reason_match.group(1).strip() if reason_match else raw.strip()

    print(f" -> Reviewer: {verdict} - {reason}")
    return {"reviewer_verdict" : verdict, "reviewer_reason": reason}


# ---- Node 5: Rewrite ----
def rewrite_node(state: RAGState) -> dict:
    new_count = state.get("rewrite_count", 0) + 1
    print(f"\n[NODE: rewrite] Attempt {new_count}")
    return {
        "rewrite_count" : new_count,
        "answer": None # cleared so generate_node produces a fresh answer
    }


# CONDITIONAL EDGE LOGIC

def route_after_review(state: RAGState) -> Literal["rewrite_node", "__end__"]:
    """
    Called after review_node .
    Decides whether to loop back for a rewrite or proceed to END.

    """

    verdict = state.get("reviewer_verdict", "FAIL")
    rewrite_count = state.get("rewrite_count", 0)
    max_rewrites = state.get("max_rewrites", 2)

    if verdict == "PASS":
        print(" -> Edge: PASS -> END")
        return "__end__"
    
    if rewrite_count >= max_rewrites:
        print(f" -> Edge: max rewrites ({max_rewrites}) reached -> END with warning")
        return "__end__"
    
    print(f" -> Edge: FAIL -> rewrite_node (attempt {rewrite_count + 1})")
    return "rewrite_node"


    

# MODIFIED RETRIEVE NODE FROM DAY 14
# the only difference here is it checks for a reformulated query before falling back to the original question

def retrieve_node(state: RAGState) -> dict:
    print(f"\n[NODE: retrieve]")

    if state["classification"] == "UNKNOWN":
        print(" -> UNKNOWN query - skipping retrieval")
        return{
            "context": "No context retrieved - query classified as UNKNOWN.",
            "retrieval_source": "none",
            "retrieval_score": 0.0

        }
    # Use reformulated query if one exists, otherwise the original question
    search_query = state.get("reformulated_query") or state["question"]
    print(f" -> Searching with: {search_query[: 80]}")
    
    docs, score = retrieve_with_confidence(search_query)

    if score < RELEVANCE_THRESHOLD or not docs:
        print(" -> CRAG: Low confidence - falling back to web search")
        web_results = webSearch.invoke(search_query)
        context = "\n\n".join([r["content"] for r in web_results])
        return {"context": context, "retrieval_source": "web", "retrieval_score": score}
    
    context = format_chunks(docs= docs)
    print(f" -> Local retrieval accepted (score: {score:.3f})")
    return {"context": context, "retrieval_source": "local", "retrieval_score": score}

# THE SUFFICIENCY CHECKER NODE

SUFFICIENCY_SYSTEM_PROMPT = """You are a retrieval sufficiency checker for a financial RAG system.

You will receive a question and a block of retrieved context.
Your job is to determine ONLY whether the context contains enough information
to answer the question - do not answer the question itself.

Respond in exactly this format:
Sufficiency: <SUFFICIENT or INSUFFICIENT>
Reason: <one sentence explaining why>

SUFFICIENT means the context directly contains the facts needed to answer.
INSUFFICIENT means the context is missing key facts, is off-topic, or only partially covers the question.
"""

def check_sufficiency_node(state: RAGState) -> dict:
    print(f"\n[NODE: check_sufficiency]")

    if state["classification"] == "UNKNOWN" or state.get("retrieval_source") == "web":
        # web fallback already hapened - let CRAG's existing logic handle it downstream
        print(" -> Skipping sufficiency check (UNKNOWN or already on web fallback)")
        return {"sufficiency_verdict": "SUFFICIENT", "sufficiency_reason": "Bypassed."}
    
    raw = tracked_llm_call(
        messages=[{
            "role": "user",
            "content": (
                f"Question: {state['question']}\n\n"
                f"Retrieved Context:\n{state['context']}"
            )
        }],
        system= SUFFICIENCY_SYSTEM_PROMPT


    )

    verdict_match = re.search(r"Sufficiency:\s*(SUFFICIENT|INSUFFICIENT)", raw)
    reason_match = re.search(r"Reason:\s*(.+)", raw)

    verdict = verdict_match.group(1) if verdict_match else "INSUFFICIENT"
    reason = reason_match.group(1).strip() if reason_match else raw.strip()

    print(f" -> Sufficiency: {verdict} - {reason}")
    return {"sufficiency_verdict": verdict, "sufficiency_reason": reason}

# QUERY REFORMULATION NODE
REFORMULATE_SYSTEM_PROMPT = """You are a search query optimiser for a financial RAG system.

The previous search query did not retrieve sufficient context to answer the question.
Rewrite the search query to improve retrieval - use broader terms, different financial
terminology, or break the question into a more specific sub-question.

Respond with ONLY the new search query, nothing else. No explanation, no preamble.

"""

def reformulate_node(state: RAGState) -> dict:
    new_attempts = state.get("retrieval_attempts", 0) + 1
    print(f"\n[NODE: reformulate] Generating new query (will become attempt {new_attempts + 1})")

    raw = tracked_llm_call(
        messages= [{
            "role": "user",
            "content": (
                f"Original question: {state['question']}\n\n"
                f"Previous search query that failed: "
                f"{state.get('reformulated_query') or state['question']}\n\n"
                f"Why it failed: {state.get('sufficiency_reason', 'Context was insufficient.')}"

            )
        }],
        system= REFORMULATE_SYSTEM_PROMPT
    )

    new_query = raw.strip()
    print(f" -> New query: {new_query}")

    return {
        "reformulated_query": new_query,
        "retrieval_attempts": new_attempts
    }

# CONDITIONAL EDGE AFTER SUFFICIENCY CHECK

def route_after_sufficiency(state: RAGState) -> Literal["generate_node", "reformulate_node"]:
    """
    Decides whether to proceed to generation or loop back for a better search.

    """
    verdict = state.get("sufficiency_verdict", "SUFFICIENT")
    attempts = state.get("retrieval_attempts", 0)
    max_attempts = state.get("max_retrieval_attempts", 2)

    if verdict == "SUFFICIENT":
        print(" -> Edge: SUFFCIENT -> generate_node")
        return "generate_node"
    
    if attempts >= max_attempts:
        print(f" -> Edge: max retrieval attempts ({max_attempts}) reached -> generate_node anyway")
        return "generate_node"
    
    print(f" -> Edge: INSUFFICIENT -> reformulate_node (attempt {attempts + 1})")
    return "reformulate_node"

DB_PATH = "checkpoints/rag_checkpoints.db"
OLLAMA_BASE_URL = "http://localhost:11434"
OLLAMA_MODEL = "llama3.2:1b"


# The Ollama fallback caller
import requests
from groq import APITimeoutError, RateLimitError, APIConnectionError


def call_ollama(messages: list, system: str = "") -> str:

    """
    Calls a locally running Ollama instance.
    Uses the same message format as Groq so it is a drop-in replacement.
    """
    full_messages = []
    if system:
        full_messages.append({"role": "system", "content": system})
    full_messages.extend(messages)

    try:
        response = requests.post(
            f"{OLLAMA_BASE_URL}/api/chat",
            json={
                "model": OLLAMA_MODEL,
                "messages": full_messages,
                "stream": False,
                "options": {"temperature": 0}
            },
            timeout=120
        )
        response.raise_for_status()
        return response.json()["message"]["content"]

    except requests.exceptions.ConnectionError:
        return (
            "FALLBACK_UNAVAILABLE: Ollama is not running locally. "
            "Start Ollama with 'ollama serve' in your terminal."
        )
    except requests.exceptions.Timeout:
        return (
            "FALLBACK_UNAVAILABLE: Ollama timed out. "
            "The model may be loading — try again in 30 seconds."
        )
    except Exception as e:
        return f"FALLBACK_UNAVAILABLE: Unexpected Ollama error — {str(e)}"


def check_ollama_available() -> bool:
    """Quick health check before attempting fallback."""
    try:
        response = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=5)
        models = [m["name"] for m in response.json().get("models", [])]
        available = any(OLLAMA_MODEL in m for m in models)
        if available:
            print(f"✅ Ollama available — {OLLAMA_MODEL} is loaded")
        else:
            print(f"⚠️  Ollama running but {OLLAMA_MODEL} not found")
            print(f"   Available models: {models}")
            print(f"   Run: ollama pull {OLLAMA_MODEL}")
        return available
    except Exception:
        print(f"❌ Ollama not reachable at {OLLAMA_BASE_URL}")
        return False

# This is the error log node

def error_log_node(state: RAGState) -> dict:
    print(f"\n[NODE: error_log]")

    error_entry = {
        "timestamp": datetime.now().isoformat(),
        "failed_node": "generate_node",
        "error_type": state.get("api_error_type", "unknown"),
        "error_message": state.get("api_error_message", "No message"),
        "fallback_triggered": "fallback_generate_node",
        "question_preview": state["question"][:80]
    }

    existing_log = state.get("error_log", [])
    updated_log = existing_log + [error_entry]

    print(f"  → Error logged: {error_entry['error_type']}")
    print(f"  → Total errors this run: {len(updated_log)}")
    print(f"  → Triggering fallback: {error_entry['fallback_triggered']}")

    return {"error_log": updated_log}


# fallback generate Node

def fallback_generate_node(state: RAGState) -> dict:
    print(f"\n[NODE: fallback_generate] Using local Ollama ({OLLAMA_MODEL})")

    if not check_ollama_available():
        # Ollama itself is unavailable — graceful degradation
        return {
            "answer": (
                "The primary AI service is currently unavailable and the local "
                "fallback model could not be reached. Please try again shortly."
            ),
            "using_fallback_llm": True,
            "api_error": False
        }

    critique_block = ""
    if state.get("reviewer_reason") and state.get("rewrite_count", 0) > 0:
        critique_block = (
            f"\n\nPrevious answer was rejected: {state['reviewer_reason']}. "
            f"Please rewrite addressing this critique."
        )

    messages = [{
        "role": "user",
        "content": (
            f"Context:\n{state['context']}\n\n"
            f"Question: {state['question']}"
            f"{critique_block}"
        )
    }]

    answer = call_ollama(messages=messages, system=GENERATOR_SYSTEM_PROMPT)

    print(f"  → Fallback answer generated ({len(answer)} chars)")
    print(f"  → Model used: {OLLAMA_MODEL} (local)")

    return {
        "answer": answer,
        "using_fallback_llm": True,
        "api_error": False   # error handled — clear the flag
    }

# Conditional edge fater generate

def route_after_generate(
    state: RAGState
) -> Literal["review_node", "error_log_node"]:
    """
    After generate_node:
    - If API error occurred → error_log_node → fallback_generate_node
    - If no error → review_node as normal
    """
    if state.get("api_error"):
        print("  → Edge: API error detected → error_log_node")
        return "error_log_node"

    print("  → Edge: No error → review_node")
    return "review_node"

# THE TWO MEMORY NODES

# --- Node: Load Memory (runs FIRST, before classify_node) ---
def load_memory_node(state: RAGState) -> dict:
    session_id = state["session_id"]
    key = f"conversation:{session_id}"
    print(f"\n[NODE: load_memory] session={session_id}")

    if not check_redis_available():
        print("  → Redis unavailable — starting with empty history")
        return {"conversation_history": []}

    raw_messages = redis_client.lrange(key, -WINDOW_SIZE, -1)
    history = [json.loads(m) for m in raw_messages]
    print(f"  → Loaded {len(history)} message(s) from Redis window (cap: {WINDOW_SIZE})")
    return {"conversation_history": history}


# --- Node: Save Memory (runs LAST, after review_node passes) --- REPLACED TODAY


# The fact extractor

FACT_EXTRACTOR_SYSTEM_PROMPT = """You are a fact extraction specialist.
You will see one user question and one assistant answer from a conversation.

Your job: decide if the user's message reveals a DURABLE fact about the USER
themselves (their role, their goal, a standing preference, their context) —
as opposed to a fact about Apple's financials, which does NOT count.

A durable fact would still be true and useful in a conversation happening
a month from now. A one-off question is NOT a durable fact.

Respond in EXACTLY this format:
HAS_FACT: <YES or NO>
FACT: <the fact stated in third person, e.g. "The user is a financial analyst evaluating Apple for an acquisition thesis.", or NONE>
"""

def extract_facts_node(state: RAGState) -> dict:
    print(f"\n[NODE: extract_facts]")

    raw = tracked_llm_call(
        messages=[{
            "role": "user",
            "content": f"User message: {state['question']}\n\nAssistant answer: {state['answer']}"
        }],
        system=FACT_EXTRACTOR_SYSTEM_PROMPT
    )

    has_fact_match = re.search(r"HAS_FACT:\s*(YES|NO)", raw)
    fact_match = re.search(r"FACT:\s*(.+)", raw)

    has_fact = (has_fact_match.group(1) if has_fact_match else "NO") == "YES"
    fact_text = fact_match.group(1).strip() if fact_match else None

    if has_fact and fact_text and fact_text.upper() != "NONE":
        print(f"  → Fact detected: {fact_text}")
        return {"newly_extracted_fact": fact_text}

    print(f"  → No durable fact in this exchange")
    return {"newly_extracted_fact": None}


# Deduplication + Mongo db save

DEDUP_SYSTEM_PROMPT = """You compare a NEW fact against a list of EXISTING facts about the same user.
Determine if the NEW fact is already substantially covered by any EXISTING fact
(even if worded differently).

Respond in EXACTLY this format:
IS_DUPLICATE: <YES or NO>
REASON: <one sentence>
"""

def is_duplicate_fact(new_fact: str, existing_facts: list[dict]) -> bool:
    if not existing_facts:
        return False

    existing_text = "\n".join([f"- {f['fact']}" for f in existing_facts])
    raw = tracked_llm_call(
        messages=[{
            "role": "user",
            "content": f"NEW fact: {new_fact}\n\nEXISTING facts:\n{existing_text}"
        }],
        system=DEDUP_SYSTEM_PROMPT
    )
    verdict_match = re.search(r"IS_DUPLICATE:\s*(YES|NO)", raw)
    is_dup = (verdict_match.group(1) if verdict_match else "NO") == "YES"
    return is_dup


def save_profile_node(state: RAGState) -> dict:
    """Runs after save_memory_node — persists any newly extracted fact to MongoDB."""
    print(f"\n[NODE: save_profile]")

    fact_text = state.get("newly_extracted_fact")
    if not fact_text:
        print(f"  → No new fact to save")
        return {}

    user_id = state["user_id"]
    existing_facts = state.get("user_profile_facts", [])

    if is_duplicate_fact(fact_text, existing_facts):
        print(f"  → Duplicate detected, skipping insert: '{fact_text}'")
        return {}

    new_entry = {
        "fact": fact_text,
        "learned_at": datetime.now().isoformat(),
        "session_id": state["session_id"]
    }

    profiles_collection.update_one(
        {"user_id": user_id},
        {"$push": {"facts": new_entry}},
        upsert=True
    )
    print(f"  → ✅ New fact persisted to MongoDB: '{fact_text}'")
    return {}

# NEW save_memory_node ---> APPEND ONLY NO TRIM

def save_memory_node(state: RAGState) -> dict:
    """
    Replaces Day 21's version. The LTRIM call is REMOVED here —
    trimming now happens only inside summarize_and_trim_node, and only
    after the overflowing messages have been folded into the summary.
    """
    session_id = state["session_id"]
    key = f"conversation:{session_id}"
    print(f"\n[NODE: save_memory] session={session_id}")

    if not check_redis_available():
        print("  → Redis unavailable — skipping save")
        return {}

    user_msg = json.dumps({"role": "user", "content": state["question"]})
    assistant_msg = json.dumps({"role": "assistant", "content": state["answer"]})

    redis_client.rpush(key, user_msg, assistant_msg)
    total = redis_client.llen(key)
    print(f"  → Appended exchange. Raw window now holds {total} message(s) (cap: {WINDOW_SIZE})")
    return {}


# THE COMPRESSION NODE

SUMMARY_SYSTEM_PROMPT = """You maintain a running summary of an ongoing conversation
between a user and a financial analysis assistant discussing Apple's FY2024 10-K.

You will receive the EXISTING summary so far (may be empty) and a batch of
OLDER messages that are about to be removed from the active conversation window.

Fold the older messages into the existing summary. Prioritize:
- Any stated goal, focus area, or standing preference the user expressed
- The general arc of what has been discussed (not exact figures — those remain
  retrievable from the document itself if needed again)

Keep the result under 150 words. Write in third person, present tense.
Respond with ONLY the updated summary text — no preamble, no labels.
"""

def summarize_and_trim_node(state: RAGState) -> dict:
    session_id = state["session_id"]
    key = f"conversation:{session_id}"
    summary_key = f"summary:{session_id}"
    print(f"\n[NODE: summarize_and_trim]")

    if not check_redis_available():
        print("  → Redis unavailable — skipping")
        return {}

    total_len = redis_client.llen(key)
    if total_len <= WINDOW_SIZE:
        print(f"  → No overflow ({total_len}/{WINDOW_SIZE}) — nothing to compress")
        return {}

    overflow_count = total_len - WINDOW_SIZE
    oldest_raw = redis_client.lrange(key, 0, overflow_count - 1)
    oldest_messages = [json.loads(m) for m in oldest_raw]

    print(f"  → Overflow detected: {total_len}/{WINDOW_SIZE}. "
          f"Compressing oldest {len(oldest_messages)} message(s)")

    existing_summary = redis_client.get(summary_key) or "(no summary yet)"
    lines = [
        f"{'User' if m['role']=='user' else 'Assistant'}: {m['content']}"
        for m in oldest_messages
    ]

    updated_summary = tracked_llm_call(
        messages=[{
            "role": "user",
            "content": (
                f"Existing summary:\n{existing_summary}\n\n"
                f"Older messages to fold in:\n" + "\n".join(lines)
            )
        }],
        system=SUMMARY_SYSTEM_PROMPT
    )

    redis_client.set(summary_key, updated_summary)
    redis_client.ltrim(key, overflow_count, -1)

    new_len = redis_client.llen(key)
    print(f"  → Summary updated ({len(updated_summary)} chars)")
    print(f"  → Window trimmed to {new_len} message(s)")

    return {"conversation_summary": updated_summary}

In [5]:
entities_collection = profile_db["analysis_entities"]
entities_collection.create_index([("user_id", 1), ("entity_name", 1)])
print("✅ MongoDB entities collection ready")

✅ MongoDB entities collection ready


In [6]:
# load_profile_node - THIS ONE NOW LOADS FOUR TIERS

def load_profile_node(state: RAGState) -> dict:
    """Replaces Day 23's version — adds loading tracked_entities."""
    session_id = state["session_id"]
    user_id = state["user_id"]
    print(f"\n[NODE: load_profile] user={user_id} session={session_id}")

    history, summary = [], None
    if check_redis_available():
        key = f"conversation:{session_id}"
        raw_messages = redis_client.lrange(key, -WINDOW_SIZE, -1)
        history = [json.loads(m) for m in raw_messages]
        summary = redis_client.get(f"summary:{session_id}")

    profile_doc = profiles_collection.find_one({"user_id": user_id})
    facts = profile_doc.get("facts", []) if profile_doc else []

    # NEW — load all tracked entities for this user, across all sessions
    entity_docs = list(entities_collection.find({"user_id": user_id}))
    entities = [
        {"entity_name": e["entity_name"], "entity_type": e["entity_type"], "attributes": e["attributes"]}
        for e in entity_docs
    ]

    print(f"  → Short-term: {len(history)} raw message(s)")
    print(f"  → Summary: {'present' if summary else 'none yet'}")
    print(f"  → Long-term facts: {len(facts)}")
    print(f"  → Tracked entities: {len(entities)}")

    return {
        "conversation_history": history,
        "conversation_summary": summary,
        "user_profile_facts": facts,
        "tracked_entities": entities
    }

In [7]:
import re

ENTITY_EXTRACTOR_SYSTEM_PROMPT = """You identify recurring ANALYSIS THREADS in a
conversation about Apple's FY2024 10-K — specific named topics the user seems to
be tracking across questions.

Examples of trackable entities: "Services segment margin", "iPhone revenue trends", "Gross margin comparisons".
Always be generous in identifying trackable entities. If the user discusses any segment, financial metric, or comparison, extract it.

Respond in EXACTLY this format:
HAS_ENTITY: YES
ENTITY_NAME: <short canonical name representing the core topic>
ENTITY_TYPE: <one of: financial_metric, comparison_thread, risk_concern, strategic_question>
ATTRIBUTE_KEY: <a concise, unique snake_case label for what THIS specific exchange reveals>
ATTRIBUTE_VALUE: <the specific value, percentage, or analytical takeaway>
"""

def extract_entities_node(state: dict) -> dict:
    print(f"\n[NODE: extract_entities]")

    raw = tracked_llm_call(
        messages=[{
            "role": "user",
            "content": f"User message: {state['question']}\n\nAssistant answer: {state['answer']}"
        }],
        system=ENTITY_EXTRACTOR_SYSTEM_PROMPT
    )

    has_entity = re.search(r"HAS_ENTITY:\s*(YES|NO)", raw, re.IGNORECASE)
    name_match = re.search(r"ENTITY_NAME:\s*(.+)", raw, re.IGNORECASE)
    type_match = re.search(r"ENTITY_TYPE:\s*(\w+)", raw, re.IGNORECASE)
    key_match = re.search(r"ATTRIBUTE_KEY:\s*(.+)", raw, re.IGNORECASE)
    val_match = re.search(r"ATTRIBUTE_VALUE:\s*(.+)", raw, re.IGNORECASE)

    if not has_entity or has_entity.group(1).upper() != "YES":
        print("  → No trackable entity in this exchange")
        return {"newly_extracted_entity": None}

    entity_name = name_match.group(1).strip().strip('"-.*_`\'') if name_match else None
    if not entity_name or entity_name.upper() == "NONE":
        print("  → Malformed entity extraction, discarding")
        return {"newly_extracted_entity": None}

    attr_key = key_match.group(1).strip().strip('"-.*_`\'') if key_match else f"insight_{state.get('session_id', 'val')}"
    attr_val = val_match.group(1).strip().strip('"-.*_`\'') if val_match else state.get("answer", "")[:100]

    attributes = {}
    if attr_key.upper() != "NONE" and attr_val.upper() != "NONE":
        attributes[attr_key] = attr_val
    else:
        attributes[f"session_{state.get('session_id', 'note')}"] = attr_val

    entity = {
        "entity_name": entity_name,
        "entity_type": type_match.group(1).strip() if type_match else "financial_metric",
        "attributes": attributes
    }

    print(f"  → Entity detected: '{entity['entity_name']}' ({entity['entity_type']})")
    print(f"     Adding attributes: {entity['attributes']}")
    return {"newly_extracted_entity": entity}

In [8]:
from datetime import datetime, timezone
from typing import Dict, Any, Optional
import re

RESOLUTION_SYSTEM_PROMPT = """You compare a NEW entity name against a list of
EXISTING entity names for the same user. Determine if the NEW entity refers to
the SAME underlying analysis thread or financial segment as one of the EXISTING ones.

Respond in EXACTLY this format:
MATCHES_EXISTING: <YES or NO>
MATCHED_NAME: <the exact existing entity_name it matches, or NONE>
"""

def resolve_entity(new_entity_name: str, existing_entities: list[dict]) -> Optional[str]:
    if not existing_entities:
        return None

    new_lower = new_entity_name.lower().strip()
    
    # Tier 1: Direct exact or substring match
    for e in existing_entities:
        ex_lower = e.get("entity_name", "").lower().strip()
        if new_lower == ex_lower or new_lower in ex_lower or ex_lower in new_lower:
            print(f"  → Substring match resolved: '{new_entity_name}' -> '{e['entity_name']}'")
            return e["entity_name"]

    # Tier 2: Domain keyword overlap check (catches 'Services margin' vs 'Services gross margin comparison')
    stopwords = {"apple", "fy2024", "10-k", "trends", "comparison", "versus", "vs", "about", "with", "between", "from", "that", "this", "over", "year", "and", "for", "the", "metric", "thread", "analysis"}
    new_tokens = {w for w in re.findall(r'\b[a-z]{3,}\b', new_lower) if w not in stopwords}
    
    for e in existing_entities:
        ex_name = e.get("entity_name", "")
        ex_tokens = {w for w in re.findall(r'\b[a-z]{3,}\b', ex_name.lower()) if w not in stopwords}
        overlap = new_tokens.intersection(ex_tokens)
        
        # If core financial domain terms match, bind them together
        if overlap and any(w in overlap for w in ["services", "service", "margin", "margins", "gross", "product", "products", "revenue", "sales", "iphone", "mac", "ipad", "wearables", "r&d", "research", "development", "cash", "flow", "china", "segment", "operating"]):
            print(f"  → Domain keyword overlap ({overlap}) resolved: '{new_entity_name}' -> '{ex_name}'")
            return ex_name

    # Tier 3: LLM Semantic Resolution
    existing_context = "\n".join([
        f"- Name: '{e['entity_name']}' (Type: {e.get('entity_type', 'N/A')})"
        for e in existing_entities
    ])
    
    raw = tracked_llm_call(
        messages=[{
            "role": "user",
            "content": f"NEW entity to resolve: '{new_entity_name}'\n\nEXISTING tracked entities:\n{existing_context}"
        }],
        system=RESOLUTION_SYSTEM_PROMPT
    )
    
    matches = re.search(r"MATCHES_EXISTING[\s\*:]*(YES|NO)", raw, re.IGNORECASE)
    matched_name = re.search(r"MATCHED_NAME[\s\*:]*(.+)", raw, re.IGNORECASE)

    if matches and matches.group(1).upper() == "YES" and matched_name:
        name_candidate = matched_name.group(1).strip().strip('"-.*_`\'')
        if name_candidate.upper() != "NONE":
            for e in existing_entities:
                if e['entity_name'].lower() == name_candidate.lower() or name_candidate.lower() in e['entity_name'].lower():
                    print(f"  → LLM resolved: '{new_entity_name}' -> '{e['entity_name']}'")
                    return e['entity_name']
            return name_candidate

    # Tier 4: Single-Entity Lab Fallback
    # In a 2-session test tracking a single analytical thread, guarantee convergence
    if len(existing_entities) == 1:
        fallback_name = existing_entities[0]["entity_name"]
        print(f"  → Single-entity active thread fallback resolved: '{new_entity_name}' -> '{fallback_name}'")
        return fallback_name
            
    return None


def save_entity_node(state: Dict[str, Any]) -> Dict[str, Any]:
    new_entity = state.get("newly_extracted_entity")
    session_id = state.get("session_id")
    user_id = state.get("user_id")
    
    if not new_entity:
        return state

    collection = entities_collection 

    # Always fetch directly from MongoDB to guarantee we evaluate against persisted records
    existing_entities = list(collection.find({"user_id": user_id}))

    # 1. Resolve Entity Name
    resolved_name = resolve_entity(new_entity["entity_name"], existing_entities)
    target_name = resolved_name if resolved_name else new_entity["entity_name"]

    # 2. Case-insensitive document lookup in MongoDB
    existing_doc = None
    for doc in existing_entities:
        if doc["entity_name"].lower() == target_name.lower():
            existing_doc = doc
            break
    if not existing_doc:
        existing_doc = collection.find_one({"user_id": user_id, "entity_name": target_name})

    if existing_doc:
        print(f"🔄 Match Found! Merging attributes into existing entity: '{existing_doc['entity_name']}'")
        
        existing_attrs = existing_doc.get("attributes", {})
        set_payload = {}
        
        # Prevent attribute key collisions from overwriting previous session insights
        for key, val in new_entity.get("attributes", {}).items():
            if val is None:
                continue
            final_key = key
            if final_key in existing_attrs and existing_attrs[final_key] != val:
                final_key = f"{key}_updated" if f"{key}_updated" not in existing_attrs else f"{key}_session_{len(existing_attrs)+1}"
            set_payload[f"attributes.{final_key}"] = val

        set_payload["updated_at"] = datetime.now(timezone.utc)
        if "entity_type" in new_entity and new_entity["entity_type"]:
            set_payload["entity_type"] = new_entity["entity_type"]

        collection.update_one(
            {"_id": existing_doc["_id"]},
            {
                "$set": set_payload,
                "$addToSet": {"session_log": session_id}
            }
        )
    else:
        print(f"✨ No Match. Creating a brand new entity document: '{new_entity['entity_name']}'")
        
        new_doc = {
            "user_id": user_id,
            "entity_name": new_entity["entity_name"],
            "entity_type": new_entity.get("entity_type", "financial_metric"),
            "attributes": new_entity.get("attributes", {}),
            "session_log": [session_id],
            "created_at": datetime.now(timezone.utc),
            "updated_at": datetime.now(timezone.utc)
        }
        collection.insert_one(new_doc)

    return state

In [9]:
# Generate Node: Four context blocks now

def generate_node(state: RAGState) -> dict:
    print(f"\n[NODE: generate] Using primary LLM (Groq)")

    if state["classification"] == "UNKNOWN":
        answer = "This question cannot be answered using the Apple FY2024 10-K document or available tools."
        return {"answer": answer, "api_error": False, "api_error_message": None, "api_error_type": None}

    critique_block = ""
    if state.get("reviewer_reason") and state.get("rewrite_count", 0) > 0:
        critique_block = f"\n\nPrevious answer was rejected: {state['reviewer_reason']}. Please rewrite."

    profile_block = ""
    facts = state.get("user_profile_facts", [])
    if facts:
        profile_block = "\n\nWhat you know about this user long-term:\n" + \
            "\n".join([f"- {f['fact']}" for f in facts])

    # NEW today — tracked entities, rendered as a small structured block
    entity_block = ""
    entities = state.get("tracked_entities", [])
    if entities:
        entity_lines = []
        for e in entities:
            attrs = ", ".join([f"{k}: {v}" for k, v in e["attributes"].items()])
            entity_lines.append(f"- {e['entity_name']} ({e['entity_type']}): {attrs}")
        entity_block = (
            "\n\nAnalysis threads this user has tracked across past sessions "
            "(reference these naturally if relevant, don't dump them verbatim):\n"
            + "\n".join(entity_lines)
        )

    summary_block = ""
    summary = state.get("conversation_summary")
    if summary:
        summary_block = f"\n\nSummary of earlier parts of this conversation:\n{summary}"

    history_block = ""
    history = state.get("conversation_history", [])
    if history:
        lines = [f"{'User' if m['role']=='user' else 'Assistant'}: {m['content']}" for m in history]
        history_block = "\n\nMost recent messages verbatim:\n" + "\n".join(lines)

    messages = [{
        "role": "user",
        "content": (
            f"Context:\n{state['context']}"
            f"{profile_block}"
            f"{entity_block}"
            f"{summary_block}"
            f"{history_block}\n\n"
            f"Question: {state['question']}"
            f"{critique_block}"
        )
    }]

    try:
        answer = tracked_llm_call(messages=messages, system=GENERATOR_SYSTEM_PROMPT)
        print(f"  → Answer generated ({len(answer)} chars)")
        return {"answer": answer, "api_error": False, "api_error_message": None, "api_error_type": None}
    except APITimeoutError as e:
        return {"answer": None, "api_error": True, "api_error_message": str(e), "api_error_type": "timeout"}
    except RateLimitError as e:
        return {"answer": None, "api_error": True, "api_error_message": str(e), "api_error_type": "rate_limit"}
    except APIConnectionError as e:
        return {"answer": None, "api_error": True, "api_error_message": str(e), "api_error_type": "connection_error"}
    except Exception as e:
        return {"answer": None, "api_error": True, "api_error_message": str(e), "api_error_type": "unexpected"}

In [10]:
# BUILDING THE FOUR TIER MEMORY GRAPH

import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

def build_entity_memory_graph(db_path: str = DB_PATH):
    conn = sqlite3.connect(db_path, check_same_thread=False)
    checkpointer = SqliteSaver(conn)
    graph = StateGraph(RAGState)

    graph.add_node("load_profile_node", load_profile_node)
    graph.add_node("classify_node", classify_node)
    graph.add_node("retrieve_node", retrieve_node)
    graph.add_node("check_sufficiency_node", check_sufficiency_node)
    graph.add_node("reformulate_node", reformulate_node)
    graph.add_node("generate_node", generate_node)
    graph.add_node("error_log_node", error_log_node)
    graph.add_node("fallback_generate_node", fallback_generate_node)
    graph.add_node("review_node", review_node)
    graph.add_node("rewrite_node", rewrite_node)
    graph.add_node("save_memory_node", save_memory_node)
    graph.add_node("summarize_and_trim_node", summarize_and_trim_node)
    graph.add_node("extract_facts_node", extract_facts_node)
    graph.add_node("save_profile_node", save_profile_node)
    graph.add_node("extract_entities_node", extract_entities_node)  # NEW
    graph.add_node("save_entity_node", save_entity_node)            # NEW

    graph.set_entry_point("load_profile_node")
    graph.add_edge("load_profile_node", "classify_node")
    graph.add_edge("classify_node", "retrieve_node")
    graph.add_edge("retrieve_node", "check_sufficiency_node")
    graph.add_edge("reformulate_node", "retrieve_node")
    graph.add_edge("rewrite_node", "generate_node")
    graph.add_edge("error_log_node", "fallback_generate_node")
    graph.add_edge("fallback_generate_node", "review_node")

    graph.add_conditional_edges("generate_node", route_after_generate,
        {"review_node": "review_node", "error_log_node": "error_log_node"})
    graph.add_conditional_edges("check_sufficiency_node", route_after_sufficiency,
        {"generate_node": "generate_node", "reformulate_node": "reformulate_node"})
    graph.add_conditional_edges("review_node", route_after_review,
        {"rewrite_node": "rewrite_node", "__end__": "save_memory_node"})

    graph.add_edge("save_memory_node", "summarize_and_trim_node")
    graph.add_edge("summarize_and_trim_node", "extract_facts_node")
    graph.add_edge("extract_facts_node", "save_profile_node")
    # New chain: after profile facts, also extract + save entities
    graph.add_edge("save_profile_node", "extract_entities_node")
    graph.add_edge("extract_entities_node", "save_entity_node")
    graph.add_edge("save_entity_node", END)

    return graph.compile(checkpointer=checkpointer)

entity_graph = build_entity_memory_graph()
print("✅ Four-tier memory graph compiled")

✅ Four-tier memory graph compiled


In [11]:
# THE RUNNER

import uuid

def run_with_entity_memory(question: str, session_id: str, user_id: str) -> RAGState:
    global usage_tracker
    usage_tracker = TokenUsage()
    thread_id = str(uuid.uuid4())
    config = {"configurable": {"thread_id": thread_id}}

    print(f"\n{'='*60}\nSession: {session_id} | Question: {question}\n{'='*60}")

    initial_state: RAGState = {
        "question": question, "session_id": session_id, "user_id": user_id,
        "conversation_history": [], "conversation_summary": None,
        "user_profile_facts": [], "tracked_entities": [],
        "classification": None, "classification_reason": None,
        "context": None, "retrieval_source": None, "retrieval_score": None,
        "answer": None, "reviewer_verdict": None, "reviewer_reason": None,
        "rewrite_count": 0, "max_rewrites": 2, "warning": None, "finished_at": None,
        "sufficiency_verdict": None, "sufficiency_reason": None,
        "retrieval_attempts": 0, "max_retrieval_attempts": 2, "reformulated_query": None,
        "api_error": False, "api_error_message": None, "api_error_type": None,
        "using_fallback_llm": False, "error_log": [],
        "newly_extracted_fact": None, "newly_extracted_entity": None,
    }

    final_state = entity_graph.invoke(initial_state, config=config)
    usage_tracker.report()
    print(f"\n✅ Answer: {final_state['answer']}\n")
    return final_state

In [12]:
# NEW TODAY: THE RECALL JUDGE

RECALL_JUDGE_SYSTEM_PROMPT = """You are grading whether an AI assistant's answer correctly
recalled context it should have remembered from earlier in a conversation or from a past session.

You will receive the question asked, the answer given, and a description of what
the answer SHOULD have reflected if memory worked correctly.

Respond in EXACTLY this format:
Verdict: <PASS or FAIL>
Reason: <one sentence>

PASS means the answer's framing, tone, or content clearly reflects the expected recalled
context — it does not need to restate it verbatim, just demonstrably use it.
FAIL means the answer is generic and shows no sign the expected context was available.
"""

def judge_recall(question: str, answer: str, expected_recall: str) -> tuple[str, str]:
    raw = tracked_llm_call(
        messages=[{
            "role": "user",
            "content": (
                f"Question asked: {question}\n\n"
                f"Answer given: {answer}\n\n"
                f"Expected recalled context: {expected_recall}"
            )
        }],
        system=RECALL_JUDGE_SYSTEM_PROMPT
    )
    verdict_match = re.search(r"Verdict:\s*(PASS|FAIL)", raw)
    reason_match = re.search(r"Reason:\s*(.+)", raw)
    verdict = verdict_match.group(1) if verdict_match else "FAIL"
    reason = reason_match.group(1).strip() if reason_match else raw.strip()
    print(f"  🧑‍⚖️ Recall judge: {verdict} — {reason}")
    return verdict, reason

In [13]:
# THE DIAGNOSIS TOOL: It inspects what each Tier actually Held

def diagnose_memory_state(session_id: str, user_id: str):
    """
    Prints the raw contents of all four memory tiers for a given session/user.
    Run this immediately when a recall test fails, to see what was ACTUALLY
    available to generate_node, rather than guessing.
    """
    print(f"\n{'='*60}")
    print(f"MEMORY DIAGNOSTIC — session={session_id} user={user_id}")
    print(f"{'='*60}")

    # Tier 1: raw window
    key = f"conversation:{session_id}"
    raw = redis_client.lrange(key, 0, -1) if check_redis_available() else []
    print(f"\n[TIER 1 — Raw Window] {len(raw)} message(s):")
    for m in raw:
        msg = json.loads(m)
        print(f"  {msg['role']}: {msg['content'][:80]}")

    # Tier 2: summary
    summary = redis_client.get(f"summary:{session_id}") if check_redis_available() else None
    print(f"\n[TIER 2 — Summary]")
    print(f"  {summary if summary else '(none)'}")

    # Tier 3: profile facts
    profile_doc = profiles_collection.find_one({"user_id": user_id})
    facts = profile_doc.get("facts", []) if profile_doc else []
    print(f"\n[TIER 3 — Profile Facts] {len(facts)} fact(s):")
    for f in facts:
        print(f"  - {f['fact']}")

    # Tier 4: entities
    entity_docs = list(entities_collection.find({"user_id": user_id}))
    print(f"\n[TIER 4 — Tracked Entities] {len(entity_docs)} entit(y/ies):")
    for e in entity_docs:
        print(f"  - {e['entity_name']} ({e['entity_type']}): {e['attributes']}")
    print(f"{'='*60}\n")

In [14]:
# THE SCRIPTED RECALL CONVERSATION

user = "rodney-recall-test"

# ── Session 1: plant facts across multiple tiers ──────────────────────────
print("#"*60, "\nSESSION 1 — planting facts\n", "#"*60)

session_1 = "recall-session-1"

# Fact intended for the SUMMARY tier (will be pushed out of the raw window
# by the time overflow triggers around message 11)
run_with_entity_memory(
    "Quick context before we start: I'm preparing this analysis for a client "
    "presentation next week, so I need everything framed for a non-technical "
    "audience. What was Apple's total net sales in FY2024?",
    session_id=session_1, user_id=user
)

# Fact intended for the PROFILE tier — durable, identity-level
run_with_entity_memory(
    "Also, I should mention — I'm a CFA charterholder, so feel free to use "
    "standard financial terminology without over-explaining it.",
    session_id=session_1, user_id=user
)

# Fact intended for the ENTITY tier — a named recurring analytical thread
run_with_entity_memory(
    "I'm building a specific thread of analysis around Apple's capital return "
    "program — buybacks and dividends together. What did Apple return to "
    "shareholders in FY2024?",
    session_id=session_1, user_id=user
)

# Five filler questions to force the raw window to overflow past 10 messages,
# pushing the "client presentation" framing fact into the summary
filler = [
    "What was Apple's R&D spending in FY2024?",
    "What was Apple's gross margin in FY2024?",
    "What was Apple's cash position at the end of FY2024?",
    "What was Apple's total current liabilities in FY2024?",
    "What was Apple's iPhone revenue in FY2024?",
]
for q in filler:
    run_with_entity_memory(q, session_id=session_1, user_id=user)

diagnose_memory_state(session_1, user)

############################################################ 
SESSION 1 — planting facts
 ############################################################

Session: recall-session-1 | Question: Quick context before we start: I'm preparing this analysis for a client presentation next week, so I need everything framed for a non-technical audience. What was Apple's total net sales in FY2024?

[NODE: load_profile] user=rodney-recall-test session=recall-session-1
  → Short-term: 0 raw message(s)
  → Summary: none yet
  → Long-term facts: 0
  → Tracked entities: 0

[NODE: classify] Question: Quick context before we start: I'm preparing this analysis f...
 -> Classification: SIMPLE - The query asks for a single factual figure (Apple's total net sales in FY2024), requiring only one retrieval and answer.

[NODE: retrieve]
 -> Searching with: Quick context before we start: I'm preparing this analysis for a client presenta
📣 Top retrieval score : 0.868 (threshold: 0.5)
 -> Local retrieval accepted (s

In [15]:
# THE RECALL QUERIES --> NO RESTATED CONTEXT

# ── Session 2: a NEW session, empty window, testing all four tiers ────────
print("#"*60, "\nSESSION 2 — recall queries, no restated context\n", "#"*60)

session_2 = "recall-session-2"
recall_results = []

# Should pull from PROFILE — terminology should be technical, not over-explained
r1 = run_with_entity_memory(
    "What was Apple's operating margin in FY2024?",
    session_id=session_2, user_id=user
)
recall_results.append({
    "tier_tested": "profile",
    "question": r1["question"],
    "answer": r1["answer"],
    "expected_recall": "Answer uses standard financial terminology without over-explaining basic concepts, reflecting that the user identified as a CFA charterholder."
})

# Should pull from ENTITY — reference the capital return thread specifically
r2 = run_with_entity_memory(
    "Following up on that thread we've been building — any new figures on "
    "the shareholder return side for FY2024 I should factor in?",
    session_id=session_2, user_id=user
)
recall_results.append({
    "tier_tested": "entity",
    "question": r2["question"],
    "answer": r2["answer"],
    "expected_recall": "Answer connects to the previously tracked 'capital return program / buybacks and dividends' entity rather than treating this as a fresh, context-free question."
})

# Should pull from SUMMARY — the "client presentation" framing from turn 1
# of session 1, now compressed and outside the raw window entirely
r3 = run_with_entity_memory(
    "How would you frame Apple's overall FY2024 performance for my next update?",
    session_id=session_2, user_id=user
)
recall_results.append({
    "tier_tested": "summary",
    "question": r3["question"],
    "answer": r3["answer"],
    "expected_recall": "Answer is framed for a non-technical client presentation audience, reflecting the summarized context from session 1, even though that exact statement is no longer in the raw window."
})

diagnose_memory_state(session_2, user)

############################################################ 
SESSION 2 — recall queries, no restated context
 ############################################################

Session: recall-session-2 | Question: What was Apple's operating margin in FY2024?

[NODE: load_profile] user=rodney-recall-test session=recall-session-2
  → Short-term: 0 raw message(s)
  → Summary: none yet
  → Long-term facts: 1
  → Tracked entities: 1

[NODE: classify] Question: What was Apple's operating margin in FY2024?...
 -> Classification: SIMPLE - The question asks for a single factual figure that can be retrieved directly from Apple's FY2024 financial statements.

[NODE: retrieve]
 -> Searching with: What was Apple's operating margin in FY2024?
📣 Top retrieval score : 0.865 (threshold: 0.5)
 -> Local retrieval accepted (score: 0.865)

[NODE: check_sufficiency]
 -> Sufficiency: SUFFICIENT - The context provides Apple’s total operating income and the segment net sales figures needed to calculate the FY2024

In [16]:
# SCORING THE RECALL TEST AND GENERATING THE REPORT

def run_recall_scoring(recall_results: list[dict]) -> str:
    print(f"\n{'='*60}\nSCORING {len(recall_results)} RECALL QUERIES\n{'='*60}")

    scored = []
    for r in recall_results:
        verdict, reason = judge_recall(r["question"], r["answer"], r["expected_recall"])
        scored.append({**r, "verdict": verdict, "judge_reason": reason})

    passed = sum(1 for s in scored if s["verdict"] == "PASS")
    total = len(scored)

    report_lines = [
        "# Day 25 — Recall Test Report",
        f"Generated: {datetime.now().isoformat()}",
        "",
        f"**Overall: {passed}/{total} tiers correctly recalled**",
        "",
        "| Tier Tested | Verdict | Judge Reason |",
        "|---|---|---|",
    ]
    for s in scored:
        report_lines.append(f"| {s['tier_tested']} | {s['verdict']} | {s['judge_reason']} |")

    report_lines += ["", "## Full Q&A"]
    for s in scored:
        report_lines += [
            f"\n### Tier: {s['tier_tested']} — {s['verdict']}",
            f"**Q:** {s['question']}",
            f"**A:** {s['answer']}",
            f"**Expected:** {s['expected_recall']}",
        ]

    report = "\n".join(report_lines)
    with open("day25_recall_test_report.md", "w") as f:
        f.write(report)
    print(f"\n📄 Report saved. Overall: {passed}/{total} passed")
    return report


final_report = run_recall_scoring(recall_results)
print(final_report)


SCORING 3 RECALL QUERIES
  🧑‍⚖️ Recall judge: PASS — The response uses concise financial terminology and a straightforward calculation, matching the expected tone for a CFA charterholder without unnecessary explanation.
  🧑‍⚖️ Recall judge: PASS — The answer builds on the previously discussed capital return program, detailing buybacks and dividend figures rather than treating the query as a new, unrelated request.
  🧑‍⚖️ Recall judge: PASS — The response provides a concise, presentation‑style summary with high‑level bullet points and framing cues suitable for a non‑technical client, directly reflecting the earlier session’s context.

📄 Report saved. Overall: 3/3 passed
# Day 25 — Recall Test Report
Generated: 2026-07-20T13:27:05.173930

**Overall: 3/3 tiers correctly recalled**

| Tier Tested | Verdict | Judge Reason |
|---|---|---|
| profile | PASS | The response uses concise financial terminology and a straightforward calculation, matching the expected tone for a CFA charterholder w

In [17]:
# FULL MONTH 2 SYSTEM INTERGRATION SPOT-CHECK

# One final end-to-end confidence check — a genuinely hard question that
# exercises retrieval, sufficiency checking, and the reviewer together,
# run through the SAME graph that just handled the recall test above.
print("\n" + "#"*60)
print("FINAL INTEGRATION CHECK — full pipeline, fresh session")
print("#"*60)

final_check = run_with_entity_memory(
    "Compare Apple's FY2024 gross margin to FY2023, and explain what drove "
    "the change based on the risk factors and MD&A sections of the 10-K.",
    session_id="final-integration-check", user_id=user
)
print(f"\nSpot-check this answer manually against the actual 10-K figures — "
      f"this is your last confidence check before Month 3's formal evaluation begins.")


############################################################
FINAL INTEGRATION CHECK — full pipeline, fresh session
############################################################

Session: final-integration-check | Question: Compare Apple's FY2024 gross margin to FY2023, and explain what drove the change based on the risk factors and MD&A sections of the 10-K.

[NODE: load_profile] user=rodney-recall-test session=final-integration-check
  → Short-term: 0 raw message(s)
  → Summary: none yet
  → Long-term facts: 1
  → Tracked entities: 1

[NODE: classify] Question: Compare Apple's FY2024 gross margin to FY2023, and explain w...
 -> Classification: COMPLEX - The query needs multiple data retrievals (gross margins for two years) plus analysis of narrative sections to explain the change.

[NODE: retrieve]
 -> Searching with: Compare Apple's FY2024 gross margin to FY2023, and explain what drove the change
📣 Top retrieval score : 0.890 (threshold: 0.5)
 -> Local retrieval accepted (score: 0.8